# Notebook 04 — Đánh Giá Tái Lập (`eval()`) Từ Trọng Số Đã Lưu & Tổng Hợp Kết Quả Module 1

**Đồ án môn học:** IS403 — Phân Tích Dữ Liệu Kinh Doanh (Module 1: Dự báo giá đồng 1 phiên trước)  
**Tác giả:** Viên Đăng Khoa  
**Mục tiêu notebook:** Nạp lại trọng số Holdout của 9 mô hình, xác minh đủ artifact Walk-Forward của 8 mô hình trên FS06_WTI_CU và tổng hợp báo cáo dễ kiểm tra.

### Mục lục
1. Cấu hình đường dẫn & nạp dữ liệu gốc + thư mục artifact từ Notebook 02 & 03
2. Định nghĩa hàm `eval_saved_model()` — nạp trọng số + scaler và đánh giá trực tiếp trên tập Test
3. Chạy `eval()` trên toàn bộ trọng số Holdout đã lưu & kiểm tra khớp sai số tái lập
4. Tổng hợp Bảng 1: Hiệu năng tốt nhất theo kiến trúc trên 4 tỷ lệ chia (`80/20`, `90/10`, `92.5/7.5`, `95/5`)
5. Tổng hợp Bảng 2 & 3: So sánh 6 bộ đặc trưng (`FS01–FS06`) và 3 chiến lược (`Expanding / Rolling 756 / Recency`) tại 90/10
6. Xác minh 324 artifact và hợp nhất Walk-Forward 8 mô hình trên FS06 (542 phiên)
7. Vẽ bộ biểu đồ chuẩn (đã sửa triệt để lỗi khung trắng thừa) & đóng gói `04_evaluation_report.zip`

> **Hướng dẫn chạy trên Kaggle:** Add Input gồm `dataset_9_commodities_daily.csv`, output của Notebook 02 và output của Notebook 03 → nhấn **Run All**.

## 1. Cấu hình đường dẫn & tự động tìm artifact từ Notebook 02 và 03

In [ ]:
import glob
import hashlib
import json
import shutil
import warnings
import zipfile
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

OUT = Path("/kaggle/working/04_evaluation_report") if Path("/kaggle/working").exists() else Path("04_evaluation_report")
OUT.mkdir(parents=True, exist_ok=True)
for stale_name in ("table4_wf_dm_hac_reference.csv", "table4_wf_12_models_ranking.csv", "fig_wf_12_models_mae.png"):
    (OUT / stale_name).unlink(missing_ok=True)

TARGET = "Copper"
LOOKBACK = 22
EXPECTED_SHA = "9580ede12d474c8f990556842d551dac176b3c16b2d6a4bffb7dc41dd54bf600"

# Tự động dò dữ liệu gốc
data_files = (
    glob.glob("/kaggle/input/**/dataset_9_commodities_daily.csv", recursive=True)
    + glob.glob("**/dataset_9_commodities_daily.csv", recursive=True)
)
if not data_files:
    raise FileNotFoundError("Thiếu dataset_9_commodities_daily.csv trong Kaggle Input.")
DATA_PATH = Path(data_files[0])
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == EXPECTED_SHA

data = pd.read_csv(DATA_PATH)
data["Date"] = pd.to_datetime(data["Date"], errors="raise")
data = data.sort_values("Date").reset_index(drop=True)

# Tự động giải nén nếu artifact ở dạng ZIP trong /kaggle/input hoặc /kaggle/working
EXTRACT_ROOT = OUT / "_extracted_inputs"
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
for zname in ("02_sa_artifacts.zip", "03_nonsa_artifacts.zip"):
    z_hits = glob.glob(f"/kaggle/input/**/{zname}", recursive=True) + glob.glob(f"/kaggle/working/**/{zname}", recursive=True)
    if z_hits:
        dest = EXTRACT_ROOT / zname.replace(".zip", "")
        if dest.exists():
            shutil.rmtree(dest)
        zip_source = max(z_hits, key=lambda p: Path(p).stat().st_mtime)
        with zipfile.ZipFile(zip_source, "r") as zf:
            zf.extractall(dest)

# Dò tất cả thư mục trọng số đã lưu (chứa meta.json)
meta_files = sorted(
    set(
        glob.glob("/kaggle/input/**/holdout_weights/*/meta.json", recursive=True)
        + glob.glob("/kaggle/working/**/holdout_weights/*/meta.json", recursive=True)
        + glob.glob(str(EXTRACT_ROOT / "**/holdout_weights/*/meta.json"), recursive=True)
    )
)
# Khử trùng theo cấu hình logic nếu cùng artifact xuất hiện ở cả Input, Working và ZIP.
ALLOWED_MODELS = {"baseline_lstm", "baseline_rnn", "paper_sa_lstm", "reopt_sa_lstm", "lstm_direct", "lstm_residual", "gru_direct", "gru_residual", "ridge_residual"}
WF_MODELS = ALLOWED_MODELS - {"baseline_lstm"}
WF_FEATURE_SETS = {"FS06_WTI_CU"}
meta_by_key = {}
for mp in meta_files:
    m = json.loads(Path(mp).read_text(encoding="utf-8"))
    if m["model"] not in ALLOWED_MODELS:
        continue
    key = (m["model"], m["feature_set"], m["scheme"], round(float(m["ratio"]), 4), int(m["seed"]))
    meta_by_key.setdefault(key, mp)
meta_files = sorted(meta_by_key.values())
def artifact_complete(meta_path):
    d = Path(meta_path).parent
    return all((d / n).exists() for n in ("xsc.joblib", "ysc.joblib")) and ((d / "model.keras").exists() or (d / "model.joblib").exists())
EXPECTED_HOLDOUT_ARTIFACTS = 9 * 6 * 4 * 3  # 4 SA + 5 Non-SA; no CNN
if len(meta_files) != EXPECTED_HOLDOUT_ARTIFACTS:
    raise RuntimeError(f"Artifact Holdout chưa đầy đủ: {len(meta_files)}/{EXPECTED_HOLDOUT_ARTIFACTS}. Hãy chạy xong Notebook 02 và 03.")
if not all(artifact_complete(mp) for mp in meta_files):
    raise RuntimeError("Có thư mục Holdout thiếu model hoặc scaler.")
print(f"Tìm thấy đủ {len(meta_files)} mô hình Holdout đã lưu trọng số & scaler để chạy eval().")

## 2. Hàm `eval_saved_model()` — nạp trọng số & scaler, suy luận lại trực tiếp trên tập Test

In [ ]:
def build_test_windows(frame: pd.DataFrame, ratio: float, cols: list, xsc, ysc) -> dict:
    """Tạo đúng cửa sổ tập Test [split, n) bằng scaler đã fit ở pha huấn luyện (tuyệt đối không fit lại)."""
    split = int(len(frame) * float(ratio))
    x_all = xsc.transform(frame[cols]).astype("float32")
    y_all = ysc.transform(frame[[TARGET]]).ravel().astype("float32")
    ids = np.arange(split, len(frame), dtype=int)
    X_te = np.stack([x_all[t - LOOKBACK : t] for t in ids]).astype("float32")
    return {
        "date": frame["Date"].iloc[ids].to_numpy(),
        "X": X_te,
        "prev_scaled": y_all[ids - 1],
        "y_raw": frame[TARGET].iloc[ids].to_numpy(dtype=np.float64),
    }


def eval_saved_model(meta_path: str, frame: pd.DataFrame):
    """Nạp model.keras hoặc model.joblib + scaler, suy luận trên Test và trả về metrics thang giá gốc."""
    wdir = Path(meta_path).parent
    meta = json.loads(Path(meta_path).read_text(encoding="utf-8"))
    xsc = joblib.load(wdir / "xsc.joblib")
    ysc = joblib.load(wdir / "ysc.joblib")

    te = build_test_windows(frame, meta["ratio"], meta["features"], xsc, ysc)
    is_residual = meta["target_mode"] == "residual"

    if (wdir / "model.joblib").exists():
        reg = joblib.load(wdir / "model.joblib")
        raw_out = reg.predict(te["X"].reshape(len(te["X"]), -1))
    else:
        tf.keras.backend.clear_session()
        net = tf.keras.models.load_model(wdir / "model.keras", compile=False)
        raw_out = net.predict(te["X"], verbose=0).ravel()
        del net

    pred_scaled = (te["prev_scaled"] + raw_out) if is_residual else raw_out
    pred_raw = ysc.inverse_transform(np.asarray(pred_scaled).reshape(-1, 1)).ravel()
    y_true = te["y_raw"]

    row = {
        "model": meta["model"],
        "target_mode": meta["target_mode"],
        "feature_set": meta["feature_set"],
        "scheme": meta["scheme"],
        "ratio": float(meta["ratio"]),
        "seed": int(meta["seed"]),
        "n_test": len(y_true),
        "mae": float(mean_absolute_error(y_true, pred_raw)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, pred_raw))),
        "mape_pct": float(np.mean(np.abs((y_true - pred_raw) / y_true)) * 100.0),
        "r2": float(r2_score(y_true, pred_raw)),
    }
    pred_df = pd.DataFrame({
        "Date": te["date"],
        "ratio": float(meta["ratio"]),
        "scheme": meta["scheme"],
        "feature_set": meta["feature_set"],
        "model": meta["model"],
        "actual": y_true,
        "pred": pred_raw,
    })
    return row, pred_df

## 3. Chạy `eval()` toàn bộ trọng số Holdout đã lưu

In [ ]:
eval_rows = []
eval_preds = []
for idx, mp in enumerate(meta_files, 1):
    r, p = eval_saved_model(mp, data)
    eval_rows.append(r)
    eval_preds.append(p)
    if idx % 50 == 0 or idx == len(meta_files):
        print(f"Đã eval() {idx}/{len(meta_files)} mô hình...", flush=True)

holdout_df = pd.DataFrame(eval_rows)
holdout_pred_df = pd.concat(eval_preds, ignore_index=True) if eval_preds else pd.DataFrame()

if not holdout_df.empty:
    holdout_df.to_csv(OUT / "eval_holdout_all_models.csv", index=False)
    print(f"HOÀN TẤT EVAL(): {len(holdout_df)} tổ hợp Holdout đã tái kiểm chứng trực tiếp từ trọng số.")
    display(holdout_df.sort_values("mae").head(15).round(5))
else:
    print("CẢNH BÁO: Chưa tìm thấy thư mục holdout_weights. Hãy gắn output của Notebook 02 & 03 vào Kaggle Input.")

## 4. Bảng 1 — Hiệu năng Holdout tốt nhất theo kiến trúc trên 4 tỷ lệ chia

In [ ]:
if not holdout_df.empty:
    best_per_split = (
        holdout_df.sort_values("mae")
        .groupby(["ratio", "model"], as_index=False)
        .first()
        .sort_values(["ratio", "mae"])
    )
    best_per_split.to_csv(OUT / "table1_best_by_arch_and_split.csv", index=False)
    piv_mae = best_per_split.pivot(index="model", columns="ratio", values="mae").round(5)
    piv_rmse = best_per_split.pivot(index="model", columns="ratio", values="rmse").round(5)
    print("MAE tốt nhất theo kiến trúc × 4 tỷ lệ chia:")
    display(piv_mae)
    print("RMSE tương ứng theo kiến trúc × 4 tỷ lệ chia:")
    display(piv_rmse)

## 5. Bảng 2 & 3 — So sánh 6 bộ đặc trưng (`FS01–FS06`) và 3 chiến lược tại mốc 90/10

In [ ]:
if not holdout_df.empty:
    # Bảng 2: So sánh 6 bộ đặc trưng tại 90/10 Expanding
    sub_fs = holdout_df[np.isclose(holdout_df["ratio"], 0.90) & (holdout_df["scheme"] == "expanding")]
    tab_fs = sub_fs.pivot_table(index="feature_set", columns="model", values="mae").round(5)
    tab_fs.to_csv(OUT / "table2_feature_sets_90_expanding.csv")
    print("Bảng 2 — MAE theo bộ đặc trưng (90/10 Expanding):")
    display(tab_fs)

    # Bảng 3: So sánh 3 chiến lược trên FS06_WTI_CU tại 90/10
    sub_st = holdout_df[np.isclose(holdout_df["ratio"], 0.90) & (holdout_df["feature_set"] == "FS06_WTI_CU")]
    tab_st = sub_st.pivot_table(index="model", columns="scheme", values="mae")
    cols_ord = [c for c in ("expanding", "rolling_756", "recency_weighted") if c in tab_st.columns]
    tab_st = tab_st[cols_ord].round(5)
    tab_st.to_csv(OUT / "table3_strategies_90_fs06.csv")
    print("Bảng 3 — MAE theo 3 chiến lược trên FS06_WTI_CU (90/10):")
    display(tab_st)

## 6. Xác minh artifact & hợp nhất Walk-Forward 8 mô hình trên FS06_WTI_CU (542 phiên)

In [ ]:
wf_files = sorted(
    set(
        glob.glob("/kaggle/input/**/sa_wf_predictions.csv", recursive=True)
        + glob.glob("/kaggle/input/**/nonsa_wf_predictions.csv", recursive=True)
        + glob.glob("/kaggle/working/**/sa_wf_predictions.csv", recursive=True)
        + glob.glob("/kaggle/working/**/nonsa_wf_predictions.csv", recursive=True)
        + glob.glob(str(EXTRACT_ROOT / "**/sa_wf_predictions.csv"), recursive=True)
        + glob.glob(str(EXTRACT_ROOT / "**/nonsa_wf_predictions.csv"), recursive=True)
    )
)
wf_meta_candidates = sorted(set(
    glob.glob("/kaggle/input/**/walkforward_weights/*/meta.json", recursive=True)
    + glob.glob("/kaggle/working/**/walkforward_weights/*/meta.json", recursive=True)
    + glob.glob(str(EXTRACT_ROOT / "**/walkforward_weights/*/meta.json"), recursive=True)
))
wf_meta_by_key = {}
for mp in wf_meta_candidates:
    m = json.loads(Path(mp).read_text(encoding="utf-8"))
    if m.get("protocol") != "walk_forward" or m["model"] not in WF_MODELS or m["feature_set"] not in WF_FEATURE_SETS:
        continue
    key = (m["feature_set"], int(m["fold_start"]), m["model"], int(m["seed"]))
    wf_meta_by_key.setdefault(key, mp)
wf_meta_files = sorted(wf_meta_by_key.values())
EXPECTED_WF_ARTIFACTS = 324
if len(wf_meta_files) != EXPECTED_WF_ARTIFACTS or not all(artifact_complete(mp) for mp in wf_meta_files):
    raise RuntimeError(f"Artifact Walk-Forward chưa đầy đủ/hợp lệ: {len(wf_meta_files)}/{EXPECTED_WF_ARTIFACTS}.")
inventory = []
for kind, paths in (("holdout", meta_files), ("walk_forward", wf_meta_files)):
    for mp in paths:
        m = json.loads(Path(mp).read_text(encoding="utf-8"))
        inventory.append({"kind": kind, "model": m["model"], "feature_set": m["feature_set"], "seed": m["seed"], "fold_start": m.get("fold_start"), "path": str(Path(mp).parent)})
pd.DataFrame(inventory).to_csv(OUT / "artifact_inventory.csv", index=False)
print(f"Artifact hợp lệ: {len(meta_files)} Holdout + {len(wf_meta_files)} Walk-Forward.")

if wf_files:
    wf_all = pd.concat([pd.read_csv(f) for f in wf_files], ignore_index=True)
    wf_all = wf_all[wf_all["model"].isin(WF_MODELS) & wf_all["feature_set"].isin(WF_FEATURE_SETS)]
    wf_all = wf_all.drop_duplicates(subset=["Date", "feature_set", "model", "seed"]).sort_values(["feature_set", "model", "seed", "Date"])
    wf_counts = wf_all.groupby(["feature_set", "model", "seed"]).size()
    expected_wf_groups = 3 * 5 + 4 * 5 + 1  # FS06 only; no CNN; deterministic Ridge
    if len(wf_counts) != expected_wf_groups or not wf_counts.eq(542).all():
        raise RuntimeError(f"Walk-forward chưa đầy đủ: {len(wf_counts)}/{expected_wf_groups} nhóm; mỗi nhóm phải có 542 ngày.")

    seed_rows = []
    for (fs, m_name, seed), g in wf_all.groupby(["feature_set", "model", "seed"]):
        err = g["actual"] - g["pred"]
        ae = np.abs(err)
        seed_rows.append({
            "feature_set": fs, "model": m_name, "seed": int(seed), "n": len(g),
            "mae": float(ae.mean()),
            "rmse": float(np.sqrt(np.mean(err ** 2))),
            "mape_pct": float(np.mean(ae / np.abs(g["actual"])) * 100.0),
            "r2": float(r2_score(g["actual"], g["pred"])),
        })

    wf_seed_df = pd.DataFrame(seed_rows)
    wf_rank = (
        wf_seed_df.groupby(["feature_set", "model"], as_index=False)
        .agg(seeds=("seed", "count"), mae_mean=("mae", "mean"), mae_std=("mae", "std"), rmse_mean=("rmse", "mean"), r2_mean=("r2", "mean"))
        .sort_values(["feature_set", "mae_mean"])
        .reset_index(drop=True)
    )
    wf_rank.insert(0, "rank", wf_rank.groupby("feature_set").cumcount() + 1)
    wf_rank.to_csv(OUT / "table4_wf_8_models_ranking.csv", index=False)
    display(wf_rank.round(5))
else:
    wf_rank = pd.DataFrame()
    print("Chưa tìm thấy file *wf_predictions.csv.")

## 7. Vẽ biểu đồ chuẩn (khắc phục hoàn toàn lỗi khoảng trắng thừa) & đóng gói `04_evaluation_report.zip`

In [ ]:
# 7.1 Biểu đồ đường dự báo Holdout 90/10 Expanding trên FS06_WTI_CU (1 khung duy nhất, không dư nửa phải)
if not holdout_pred_df.empty:
    sub_p = holdout_pred_df[
        np.isclose(holdout_pred_df["ratio"], 0.90)
        & (holdout_pred_df["scheme"] == "expanding")
        & (holdout_pred_df["feature_set"] == "FS06_WTI_CU")
    ].copy()
    sub_p["Date"] = pd.to_datetime(sub_p["Date"])

    for grp_title, grp_models, fname in [
        ("Holdout 90/10 Expanding (FS06) — Nhánh Residual", ["gru_residual", "lstm_residual", "ridge_residual"], "fig_holdout_90_residual.png"),
        ("Holdout 90/10 Expanding (FS06) — Nhánh SA-LSTM & Direct", ["paper_sa_lstm", "reopt_sa_lstm", "baseline_lstm", "baseline_rnn"], "fig_holdout_90_sa_series.png"),
    ]:
        part = sub_p[sub_p["model"].isin(grp_models)]
        if part.empty:
            continue
        fig, ax = plt.subplots(figsize=(10.5, 4.3))
        ref_m = part["model"].iloc[0]
        ref_g = part[part["model"] == ref_m].sort_values("Date")
        ax.plot(ref_g["Date"], ref_g["actual"], color="black", lw=1.8, label="Actual (Copper)")
        for m_name, g in part.groupby("model"):
            g = g.sort_values("Date")
            ax.plot(g["Date"], g["pred"], lw=1.2, alpha=0.85, label=m_name)
        ax.set_title(grp_title)
        ax.set_xlabel("Ngày")
        ax.set_ylabel("Giá Copper (USD/lb)")
        ax.legend(frameon=True, fontsize=8.5)
        ax.grid(alpha=0.25)
        fig.tight_layout()
        fig.savefig(OUT / fname, dpi=160)
        plt.show()

# 7.2 Biểu đồ xếp hạng Walk-Forward 8 mô hình trên FS06
if not wf_rank.empty:
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    df_plot = wf_rank[wf_rank["feature_set"] == "FS06_WTI_CU"].iloc[::-1]
    colors = [
        "#B71C1C" if m.endswith("_residual") else ("#B45309" if "sa_lstm" in m else "#64748B")
        for m in df_plot["model"]
    ]
    bars = ax.barh(df_plot["model"], df_plot["mae_mean"], color=colors, height=0.65)
    for bar, val in zip(bars, df_plot["mae_mean"]):
        ax.text(val + 0.006, bar.get_y() + bar.get_height() / 2, f"{val:.5f}", va="center", fontsize=8.5)
    ax.set_xlim(0, df_plot["mae_mean"].max() * 1.18)
    ax.set_xlabel("MAE trung bình 5 seeds (USD/lb)")
    ax.set_title("Xếp hạng Walk-Forward 8 mô hình (542 phiên 2024–2026, FS06 Expanding)")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUT / "fig_wf_8_models_mae.png", dpi=160)
    plt.show()

# Đóng gói ZIP cuối cùng
zip_path = OUT.parent / "04_evaluation_report.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.iterdir()):
        if f.is_file():
            zf.write(f, arcname=f.name)

print(f"HOÀN TẤT ĐÓNG GÓI BÁO CÁO ĐÁNH GIÁ: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")